# IMPORT n Download

## Data Loading

This cell loads the crypto market data and organizes it by timeframe.

`load_data()` returns a dictionary of OHLCV panels already resampled at multiple frequencies, together with the list of available assets. We then extract three datasets:
- `h1_data` for 1-hour bars
- `h4_data` for 4-hour bars
- `daily_data` for 1-day bars

These panel datasets are the raw market inputs used downstream for indicator construction, signal generation, target definition, and model training.


In [ ]:
from download import (
    load_data,
    load_indicator_mapping,
)

from features import (
    compute_all_indicators
)

from preprocessing import (
    add_all_signal_features,
    build_default_indicator_family_map,
    get_added_signal_columns,
    build_regression_dataset, 
    validate_regression_dataset

)

from target import (
    add_targets_for_multiple_horizons, 
    validate_target_alignment

)

from modele import (
    temporal_train_val_test_split, 
    summarize_temporal_split,
    temporal_train_val_test_split,
    train_validate_penalized_logit_grid,
    fit_penalized_logit,
    score_penalized_logit,
    extract_penalized_logit_coefficients,
    evaluate_top_k_hit_rate,
    run_penalized_logistic_rolling_backtest
    
)
from vis import (
    plot_confusion_matrix_for_backtest,
)

from modele_rf import (
    fit_random_forest_classifier,
    score_random_forest_classifier,
    train_validate_random_forest_grid,
    extract_random_forest_feature_importance,
    run_random_forest_rolling_backtest,
    evaluate_top_k_hit_rate,
)


import pandas as pd
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, log_loss, brier_score_loss


pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

data, assets = load_data()
h1_data = data["1H"]
h4_data = data["4H"]
daily_data = data["1D"]

## Indicator Metadata

This cell loads the indicator metadata table.

`load_indicator_mapping()` imports the indicator reference file, which links each indicator to its signal family and natural direction. This mapping is used later to organize features consistently across the pipeline and to assign family-specific signal construction rules.


In [ ]:
indicator_mapping = load_indicator_mapping()


# Features enginerening 

## Technical Indicator Computation

This cell computes the full technical indicator set on the daily panel.

`compute_all_indicators(daily_data)` transforms the raw OHLCV data into a broad feature matrix of technical indicators, covering trend, momentum, volatility, band/channel, and volume-based measures. The output is a panel of engineered features aligned by date and asset, ready for signal construction.


In [ ]:
features = compute_all_indicators(daily_data)

## Family-Specific Signal Engineering

This block converts raw indicator values into a standardized signal representation usable in cross-sectional ML.

We first build `indicator_family_map`, which assigns each feature column to a signal family. We then apply `add_all_signal_features(...)`, which creates four core signal variables for every eligible feature:
- `signal_direction`
- `signal_active`
- `signal_strength`
- `signal_signed_value`

The transformation is family-specific and follows the explicit rules implemented in `preprocessing.py`.

  1. Centered Oscillators

This family includes indicators naturally centered around zero, such as MACD, PPO, ROC, APO, momentum, KST, BOP, and similar oscillators.

For a raw feature `X(i,t)`:
- `signal_direction = sign(X(i,t))`
- we compute a cross-sectional **leave-one-out z-score** at each date `t`, across assets:
  `z_loo(i,t) = [X(i,t) - mu_loo(i,t)] / sigma_loo(i,t)`
- `signal_active = 1` if `abs(z_loo(i,t)) > 2.0`, else `0`
- `signal_strength = abs(z_loo(i,t))`
- `signal_signed_value = z_loo(i,t)`

So activation captures whether the oscillator is unusually extreme relative to the cross-section at that date, not just whether it is positive or negative.

   2. Extreme-Zone Oscillators

This family covers bounded or range-based oscillators such as RSI, RSX, Stochastic, MFI, CCI, PGO, COG, and the emotion / willingness proxies.

Each feature is assigned explicit bounds, for example:
- RSI / RSX / Stochastic / MFI: `[0, 100]`
- CCI: `[-200, 200]`
- PGO: `[-3, 3]`
- COG: `[-10, 10]`

For a bounded feature `X(i,t)`:
- values are clipped to the configured interval if needed
- the midpoint is:
  `m = (lower_bound + upper_bound) / 2`
- `signal_direction = +1` in the lower half, `-1` in the upper half
- `signal_active = 1` if the raw value is valid and lies inside the configured interval
- `signal_strength = abs(X_clipped(i,t) - m)`
- a signed force is defined as:
  `signed_force(i,t) = m - X_clipped(i,t)`
  so it is positive in the lower half and negative in the upper half
- `signal_signed_value` is the cross-sectional leave-one-out z-score of `signed_force(i,t)`

Hence, this family combines within-range location with cross-sectional standardization of how extreme that location is across assets.

   3. Trend Filters

This family includes features such as moving-average spreads, Aroon-style trend measures, Ichimoku trend features, linear-regression trend slopes, PSAR distance, and similar directional state variables.

For each feature `X(i,t)`, the signal is built from its time-series change:
`delta_X(i,t) = X(i,t) - X(i,t-1)`

Then:
- we compute the cross-sectional leave-one-out z-score of `delta_X(i,t)`
- this z-score is the **trend score**
- `signal_direction = sign(trend_score)`
- `signal_active = 1` if `abs(trend_score) > 2.0`
- `signal_strength = abs(trend_score)`
- `signal_signed_value = trend_score`

This means the model does not directly use the level of the trend filter as signal state, but rather whether the trend measure is strengthening or weakening relative to the cross-section.

   4. Volume Confirmation Indicators

This family includes OBV, PVT, CMF, accumulation/distribution features, and similar volume-confirmation measures.

The logic is also based on first differences:
`delta_X(i,t) = X(i,t) - X(i,t-1)`

Then:
- `signal_direction = sign(delta_X(i,t))`
- we compute the cross-sectional leave-one-out z-score of `delta_X(i,t)`
- this z-score acts as a **variance / abnormality score**
- `signal_active = 1` if `abs(z_loo(i,t)) > 2.0`
- `signal_strength = abs(z_loo(i,t))`
- `signal_signed_value = z_loo(i,t)`

So the signal captures whether the recent change in a volume-confirmation feature is unusually large relative to the asset cross-section.

  5. Band / Channel / Level Signals

This family is only explicitly implemented for **Bollinger Bands** at this stage.

The signal is built from:
- the reference price, by default `close`
- `bb_upper_*`
- `bb_lower_*`
- the corresponding Bollinger feature

For each date and asset:
- `signal_direction = +1` if price is above the upper band
- `signal_direction = -1` if price is below the lower band
- `signal_direction = 0` if price is inside the band

Distance to the nearest violated band is then computed:
- above upper band: `price - upper`
- below lower band: `lower - price`
- inside band: `0`

Then:
- a cross-sectional leave-one-out dispersion is computed on that distance
- `band_score = distance / cs_std_loo`
- `signal_active = 1` if price is outside the band and `band_score > 2.0`
- `signal_strength = abs(band_score)`
- `signal_signed_value = signal_direction * abs(band_score)`

So activation only occurs when the band break is both directional and unusually large cross-sectionally.

   6. Context Filters

Indicators in the `context_filter` family are **not converted into directional signals**. They are kept as contextual state variables and later enter the regression dataset as common explanatory features at the `date x asset` level.

This includes variables such as ATR, ADX, choppiness, entropy, kurtosis, midpoint / midprice, quantile rank, and related non-directional market-state descriptors.

  Output

Finally, `get_added_signal_columns(signals)` returns the full list of generated signal columns. This is useful for auditing which transformed signal variables were created and passed downstream into target construction and model estimation.


In [ ]:
indicator_family_map = build_default_indicator_family_map(features)

signals = add_all_signal_features(
    features,
    indicator_family_map=indicator_family_map,
)

added_cols = get_added_signal_columns(signals)

## Target Construction Across Multiple Horizons

This step builds the supervised learning targets from the signal panel.

`add_targets_for_multiple_horizons(...)` creates horizon-specific future-return targets for each signal feature, using three forecast horizons:
- `1d`
- `4d`
- `24d`

For each asset, date, indicator, and horizon:
- the signal is observed at date `t`
- the future return is computed from `t` to `t+h`
- the target remains attached to the row at date `t`

The target is built using:
- `target_scaling="cross_sectional_future_return"`
- `tau=0`

This means the raw future return is first standardized cross-sectionally at each date and horizon:
- compute `future_return(i,t,h)`
- compute the cross-sectional standard deviation across assets at the same `t` and `h`
- define `future_return_cs_scaled(i,t,h) = future_return(i,t,h) / sigma_cs(t,h)`

The binary target is then:
- `target = 1` if `signal_direction(i,t) * future_return_cs_scaled(i,t,h) > 0`
- `target = 0` otherwise

So with `tau=0`, a signal is labeled successful whenever its direction matches the sign of the cross-sectionally scaled future move.

Additional settings:
- `inactive_policy="nan"`:
  rows where `signal_active = 0` are not forced into class `0`; their target is set to missing and can be excluded later
- `min_cs_assets=3`:
  at least three valid assets are required to compute a cross-sectional dispersion
- `min_cs_std=1e-12`:
  protects against degenerate cross-sectional volatility when scaling future returns

This step is critical because it defines the prediction problem consistently across horizons while preserving the correct temporal alignment between signal date and realized forward performance.

In [ ]:
targets = add_targets_for_multiple_horizons(
    signals,
    horizons={"1d": 1, "4d": 4, "24d": 24},
    signal_columns=None,
    price_col="close",
    asset_col="asset",
    datetime_col="date",
    target_scaling="cross_sectional_future_return",
    tau=0,
    inactive_policy="nan",
    min_cs_assets=3,
    min_cs_std=1e-12,
)

## Final Regression Dataset

This step converts the wide signal panel into the final long-format supervised dataset used for modeling.

`build_regression_dataset(...)` reshapes the data so that each row corresponds to a unique:
`date x asset x indicator_name x horizon`

Each observation contains:
- the indicator identity (`indicator_name`)
- its family (`signal_family`)
- the signal variables:
  - `signal_direction`
  - `signal_active`
  - `signal_strength`
  - `signal_signed_value`
- the corresponding horizon-specific `target`
- the asset-date context features from the `context_filter` family

With:
- `include_inactive=False`, only active signals are kept
- `drop_missing_target=True`, rows without valid targets are removed
- `exclude_vix_context=True`, VIX-related context variables are excluded

This is the key reshape that turns the engineered panel into a model-ready classification dataset.

`validate_regression_dataset(...)` then checks that the result is structurally sound:
- required columns are present
- there are no duplicates on `date, asset, indicator_name, horizon`
- the number of rows, assets, indicators, horizons, and context features is coherent

At this point, `regression_df` is the final dataset used for logistic regression and random forest benchmarking.


In [ ]:
regression_df = build_regression_dataset(
    targets,
    asset_col="asset",
    datetime_col="date",
    include_inactive=False,
    drop_missing_target=True,
    exclude_vix_context=True,
)

summary = validate_regression_dataset(
    regression_df,
    asset_col="asset",
    datetime_col="date",
)

In [ ]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

display(regression_df)


# Modele 

## Penalized Logistic Regression 

### Penalized Logistic Regression with classical standardisation 

The penalized logistic regression model is used as a cross-sectional benchmark to estimate whether a given technical indicator is likely to generate a successful signal for a specific crypto asset, at a specific date, and for a specific forecast horizon. The final modeling dataset is built in long format, so each row corresponds to one `date × asset × indicator_name × horizon` observation with a binary target.

A separate model is trained for each horizon (`1d`, `4d`, `24d`) because the prediction problem changes materially with the holding period: short-horizon and longer-horizon signal dynamics are not expected to share the same structure. The model takes as inputs the signal features derived from each indicator, together with contextual market-state variables, and is evaluated under a rolling-window backtest with embargo to preserve temporal integrity.

In this implementation, the model uses **classical standardization** for numerical inputs. Concretely, numerical features are scaled with a `StandardScaler` fitted **only on the training fold** inside the sklearn pipeline, and then applied unchanged to the corresponding out-of-sample test fold. This avoids leakage while keeping the logistic coefficients numerically stable and comparable across folds.

Its most important output is not the binary class itself, but the estimated `pred_proba_success`. This probability is then used to rank indicators within each `date × asset × horizon` group, allowing the pipeline to identify which indicators appear most promising at each decision point.

1D

In [ ]:
regression_df_all = regression_df.copy()

regression_df_1d_all = regression_df_all[regression_df_all["horizon"] == "1d"].copy()

results_all = run_penalized_logistic_rolling_backtest(
    regression_df_1d_all,
    datetime_col="date",
    asset_col="asset",
    indicator_col="indicator_name",
    family_col="signal_family",
    horizon_col="horizon",
    target_col="target",
    horizon_steps_map={"1d": 1},
    period_train=365,
    period_test=30,
    period_embargo=5,
    penalty_type="l2",
    C=0.05,
    max_iter=3000,
    scaling_mode="classical",
    include_horizon=False,
    filter_active_only=True,
    verbose=0,
)

print("Accuracy moyen:", results_all["accuracy_scores"].mean())
print("F1 moyen:", results_all["f1_scores"].mean())
print("ROC AUC moyen:", results_all["roc_auc_scores"].mean())
print("Log loss moyen:", results_all["log_loss_scores"].mean())

results_all_df = results_all["oos_predictions_df"]
oos_all = results_all["oos_predictions_df"].copy()

plot_confusion_matrix_for_backtest(
    results_all,
    normalize="true",
    title="Confusion Matrix OOS - normalisée | 1d full universe",
)


4D

In [ ]:
regression_df_all = regression_df.copy()

regression_df_4d_all = regression_df_all[regression_df_all["horizon"] == "4d"].copy()

results_all_4d = run_penalized_logistic_rolling_backtest(
    regression_df_4d_all,
    datetime_col="date",
    asset_col="asset",
    indicator_col="indicator_name",
    family_col="signal_family",
    horizon_col="horizon",
    target_col="target",
    horizon_steps_map={"4d": 4},
    period_train=365,
    period_test=30,
    period_embargo=5,
    penalty_type="l2",
    C=0.05,
    max_iter=3000,
    scaling_mode="classical",
    include_horizon=False,
    filter_active_only=True,
    verbose=0,
)

print("Accuracy moyen:", results_all_4d["accuracy_scores"].mean())
print("F1 moyen:", results_all_4d["f1_scores"].mean())
print("ROC AUC moyen:", results_all_4d["roc_auc_scores"].mean())
print("Log loss moyen:", results_all_4d["log_loss_scores"].mean())

results_all_4d_df = results_all_4d["oos_predictions_df"]
oos_all_4d = results_all_4d["oos_predictions_df"].copy()

plot_confusion_matrix_for_backtest(
    results_all_4d,
    normalize="true",
    title="Confusion Matrix OOS - normalisée | 4d full universe",
)


24D

In [ ]:
regression_df_all = regression_df.copy()

regression_df_24d_all = regression_df_all[regression_df_all["horizon"] == "24d"].copy()

results_all_24d = run_penalized_logistic_rolling_backtest(
    regression_df_24d_all,
    datetime_col="date",
    asset_col="asset",
    indicator_col="indicator_name",
    family_col="signal_family",
    horizon_col="horizon",
    target_col="target",
    horizon_steps_map={"24d": 24},
    period_train=365,
    period_test=30,
    period_embargo=5,
    penalty_type="l2",
    C=0.05,
    max_iter=3000,
    scaling_mode="classical",
    include_horizon=False,
    filter_active_only=True,
    verbose=0,
)

print("Accuracy moyen:", results_all_24d["accuracy_scores"].mean())
print("F1 moyen:", results_all_24d["f1_scores"].mean())
print("ROC AUC moyen:", results_all_24d["roc_auc_scores"].mean())
print("Log loss moyen:", results_all_24d["log_loss_scores"].mean())

results_all_24d_df = results_all_24d["oos_predictions_df"]
oos_all_24d = results_all_24d["oos_predictions_df"].copy()

plot_confusion_matrix_for_backtest(
    results_all_24d,
    normalize="true",
    title="Confusion Matrix OOS - normalisée | 24d full universe",
)


### Penalized Logistic Regression with cross sectionnal standarisation 

1D

In [ ]:
regression_df_all = regression_df.copy()

regression_df_1d_all = regression_df_all[regression_df_all["horizon"] == "1d"].copy()

results_all_1d_cs = run_penalized_logistic_rolling_backtest(
    regression_df_1d_all,
    datetime_col="date",
    asset_col="asset",
    indicator_col="indicator_name",
    family_col="signal_family",
    horizon_col="horizon",
    target_col="target",
    horizon_steps_map={"1d": 1},
    period_train=365,
    period_test=30,
    period_embargo=5,
    penalty_type="l2",
    C=0.05,
    max_iter=3000,
    scaling_mode="cross_sectional",
    include_horizon=False,
    filter_active_only=True,
    verbose=0,
)

print("Accuracy moyen:", results_all_1d_cs["accuracy_scores"].mean())
print("F1 moyen:", results_all_1d_cs["f1_scores"].mean())
print("ROC AUC moyen:", results_all_1d_cs["roc_auc_scores"].mean())
print("Log loss moyen:", results_all_1d_cs["log_loss_scores"].mean())

results_all_1d_cs_df = results_all_1d_cs["oos_predictions_df"]
oos_all_1d_cs = results_all_1d_cs["oos_predictions_df"].copy()

plot_confusion_matrix_for_backtest(
    results_all_1d_cs,
    normalize="true",
    title="Confusion Matrix OOS - normalisée | 1d full universe | cross-sectional scaling",
)


4D

In [ ]:
regression_df_all = regression_df.copy()

regression_df_4d_all = regression_df_all[regression_df_all["horizon"] == "4d"].copy()

results_all_4d_cs = run_penalized_logistic_rolling_backtest(
    regression_df_4d_all,
    datetime_col="date",
    asset_col="asset",
    indicator_col="indicator_name",
    family_col="signal_family",
    horizon_col="horizon",
    target_col="target",
    horizon_steps_map={"4d": 4},
    period_train=365,
    period_test=30,
    period_embargo=5,
    penalty_type="l2",
    C=0.05,
    max_iter=3000,
    scaling_mode="cross_sectional",
    include_horizon=False,
    filter_active_only=True,
    verbose=0,
)

print("Accuracy moyen:", results_all_4d_cs["accuracy_scores"].mean())
print("F1 moyen:", results_all_4d_cs["f1_scores"].mean())
print("ROC AUC moyen:", results_all_4d_cs["roc_auc_scores"].mean())
print("Log loss moyen:", results_all_4d_cs["log_loss_scores"].mean())

results_all_4d_cs_df = results_all_4d_cs["oos_predictions_df"]
oos_all_4d_cs = results_all_4d_cs["oos_predictions_df"].copy()

plot_confusion_matrix_for_backtest(
    results_all_4d_cs,
    normalize="true",
    title="Confusion Matrix OOS - normalisée | 4d full universe | cross-sectional scaling",
)


24D

In [ ]:
regression_df_all = regression_df.copy()

regression_df_24d_all = regression_df_all[regression_df_all["horizon"] == "24d"].copy()

results_all_24d_cs = run_penalized_logistic_rolling_backtest(
    regression_df_24d_all,
    datetime_col="date",
    asset_col="asset",
    indicator_col="indicator_name",
    family_col="signal_family",
    horizon_col="horizon",
    target_col="target",
    horizon_steps_map={"24d": 24},
    period_train=365,
    period_test=30,
    period_embargo=5,
    penalty_type="l2",
    C=0.05,
    max_iter=3000,
    scaling_mode="cross_sectional",
    include_horizon=False,
    filter_active_only=True,
    verbose=0,
)

print("Accuracy moyen:", results_all_24d_cs["accuracy_scores"].mean())
print("F1 moyen:", results_all_24d_cs["f1_scores"].mean())
print("ROC AUC moyen:", results_all_24d_cs["roc_auc_scores"].mean())
print("Log loss moyen:", results_all_24d_cs["log_loss_scores"].mean())

results_all_24d_cs_df = results_all_24d_cs["oos_predictions_df"]
oos_all_24d_cs = results_all_24d_cs["oos_predictions_df"].copy()

plot_confusion_matrix_for_backtest(
    results_all_24d_cs,
    normalize="true",
    title="Confusion Matrix OOS - normalisée | 24d full universe | cross-sectional scaling",
)


### Interpretation of the logistic regression under both scaling choices

Taken together, the two scaling approaches show that the penalized logistic regression does capture a **real, albeit modest, predictive structure**. The model is not strong enough to be presented as a high-performance classifier, but it does provide a **coherent first benchmark** and a useful basis for indicator ranking.

 Main takeaway

Across both preprocessing choices, the results support three encouraging conclusions:

1. **The model is not purely random.**  
   ROC AUC remains slightly above 0.5 across all horizons, which suggests that the features contain some exploitable information.

2. **The horizon split is clearly justified.**  
   The behavior of the model changes across `1d`, `4d`, and `24d`, which confirms that indicator usefulness is horizon-dependent.

3. **The logistic regression is useful as a benchmark and ranking tool.**  
   Even when classification performance remains limited, the model still produces probabilities that can be used to order indicators within each `date × asset × horizon` decision set.

---

   Comparison of the two scaling approaches

    Classical scaling

Classical scaling delivers the **strongest overall discrimination**.  
It tends to produce slightly better ROC AUC across the three horizons and remains the most solid choice if the objective is to maximize the global predictive quality of the logistic benchmark.

Its main strength is stability: it gives the cleanest overall baseline and performs relatively better at the medium and long horizons.

   Cross-sectional scaling

Cross-sectional scaling is **more aligned with the logic of the project**, since the decision problem is fundamentally cross-sectional: at a given date, the goal is to compare indicators across assets rather than model absolute levels in isolation.

Empirically, this scaling does not dominate the classical version, but it remains interesting because:
- it is more consistent with the ranking objective,
- it improves the short-horizon classification balance,
- and it gives the best short-horizon F1 score among the two approaches.

So even if it is not uniformly better, it is methodologically attractive, especially for the short-term use case.

---

   Horizon-by-horizon interpretation

    1d horizon

This is the most promising horizon overall.

- With **classical scaling**, the model achieves slightly better discrimination.
- With **cross-sectional scaling**, the model achieves a better F1 and a more active detection of positives.

This suggests that at short horizon, the signal is weak but present, and the model is already capable of producing a ranking that is at least directionally informative. In practice, `1d` appears to be the horizon where the logistic regression is most naturally usable as a first ranking layer.

    4d horizon

This is the least convincing horizon.

The model remains above pure randomness, but only marginally. Both versions struggle here, and cross-sectional scaling does not improve the results. This suggests that the intermediate horizon is more difficult to capture with a linear model, likely because the relationship between signals and outcomes becomes less direct and more conditional on market context.

   24d horizon

This horizon is the most unstable but also the most interesting conceptually.

The classical version gives the best average AUC among the three horizons, which indicates that some longer-term structure may be present. At the same time, fold-to-fold variation remains large, which means that this structure is not stable through time.

So the `24d` model cannot yet be considered reliable, but it does suggest that longer-horizon signal usefulness may exist in certain regimes and deserves further investigation.

---

    What is encouraging in the project context

Even without overstating the results, the model already provides several useful contributions:

- it confirms that **indicator effectiveness is horizon-specific**,
- it delivers a **transparent and interpretable baseline**,
- it generates a **probability of success**, not just a binary class,
- and this probability can already be used to **rank indicators** for a given asset, date, and horizon.

This last point is important: in the context of the project, the value of the model is not only in raw classification performance, but also in its ability to structure the decision process and highlight the most promising indicators.

---

    Main limitations

The limitations remain clear and should be stated explicitly:

- discrimination remains weak overall,
- probability quality is still limited,
- performance varies substantially across folds,
- and a linear model is likely too restrictive to capture all relevant interactions between signals, families, contexts, and horizons.

In other words, this logistic regression should be seen as a **useful baseline**, not as a final model.

---

    Balanced conclusion

The results are modest, but they are still meaningful.  
They show that the pipeline is capturing a small amount of genuine structure and that the modeling framework is directionally sound.

A fair conclusion is therefore:

- **classical scaling gives the strongest overall benchmark**,  
- **cross-sectional scaling is the most conceptually aligned with the project**, especially at `1d`,  
- and the logistic regression already serves as a credible first model for **probability-based ranking of indicators**, even if stronger nonlinear models will likely be needed to unlock more predictive power.

## Random Forest 

This section runs the Random Forest benchmark separately on the three prediction horizons:
- `1d`
- `4d`
- `24d`

Each run uses the same final regression dataset, filtered to one horizon at a time, and applies the same rolling-window backtest with a `365`-day training window, a `30`-day test window, and a `5`-day embargo. This keeps the evaluation protocol fully comparable across horizons and with the logistic regression benchmark.

The Random Forest is specified as a moderately regularized non-linear model:
- `n_estimators = 400`
- `max_depth = 6`
- `min_samples_leaf = 20`
- `max_features = "sqrt"`
- `class_weight = "balanced"`

This setup allows the model to capture non-linear patterns and feature interactions while controlling tree complexity and reducing the risk of overfitting.

For each horizon, the backtest returns three main outputs:
- `global_metrics`, which summarize aggregate out-of-sample performance over the full rolling evaluation
- `fold_metrics_df`, which reports fold-by-fold results and allows performance stability to be assessed through time
- `oos_predictions_df`, which contains row-level out-of-sample predictions, including `pred_proba_success`, `pred_class`, and the ranking variables used to compare indicators within each `date × asset × horizon` group

As with the logistic regression, the key output is the estimated probability of success. This is what allows the Random Forest to be used not only as a classifier, but also as a ranking model for identifying the most promising indicators at each decision point.


In [ ]:
rf_results = run_random_forest_rolling_backtest(
    regression_df_1d_all,
    datetime_col="date",
    asset_col="asset",
    indicator_col="indicator_name",
    family_col="signal_family",
    horizon_col="horizon",
    target_col="target",                    
    horizon_steps_map={"1d": 1},
    period_train=365,
    period_test=30,
    period_embargo=5,
    n_estimators=400,
    max_depth=6,
    min_samples_leaf=20,
    max_features="sqrt",
    class_weight="balanced",
    filter_active_only=True,
    verbose=0,
)

print(rf_results["global_metrics"])

In [ ]:
rf_results = run_random_forest_rolling_backtest(
    regression_df_4d_all,
    datetime_col="date",
    asset_col="asset",
    indicator_col="indicator_name",
    family_col="signal_family",
    horizon_col="horizon",
    target_col="target",
    horizon_steps_map={"4d": 4},
    period_train=365,
    period_test=30,
    period_embargo=5,
    n_estimators=400,
    max_depth=6,
    min_samples_leaf=20,
    max_features="sqrt",
    class_weight="balanced",
    filter_active_only=True,
    verbose=0,
)

print(rf_results["global_metrics"])

In [ ]:
rf_results = run_random_forest_rolling_backtest(
    regression_df_24d_all,  
    datetime_col="date",
    asset_col="asset",
    indicator_col="indicator_name",
    family_col="signal_family",
    horizon_col="horizon",
    target_col="target",
    horizon_steps_map={"24d": 24},
    period_train=365,
    period_test=30,
    period_embargo=5,
    n_estimators=400,
    max_depth=6,
    min_samples_leaf=20,
    max_features="sqrt",
    class_weight="balanced",
    filter_active_only=True,
    verbose=0,
)

print(rf_results["global_metrics"])

Interprétation des résultats globaux par horizon

Les résultats montrent que le modèle capte un **signal faible mais réel** sur les trois horizons, sans pour autant atteindre un niveau de performance élevé. 

    Horizon 1d

Pour `1d`, les métriques sont :

- `n_obs = 858987`
- `target_rate = 0.4999`
- `accuracy = 0.5109`
- `f1 = 0.4955`
- `roc_auc = 0.5160`
- `log_loss = 0.6928`
- `brier_score = 0.2498`

La target est presque parfaitement équilibrée, ce qui rend le problème difficile à battre nettement avec une baseline simple. Le `ROC AUC` légèrement supérieur à `0.5` montre que le modèle ne se réduit pas à du hasard pur : il y a bien un peu d’information exploitable. En revanche, l’`accuracy` et le `F1` restent modestes, ce qui signifie que la séparation entre succès et échec reste faible. Le `log loss` et le `Brier score`, très proches de leurs niveaux non informatifs, indiquent aussi que les probabilités produites restent peu tranchées. Malgré cela, `1d` reste l’horizon le plus naturel pour une utilisation du modèle comme **outil de ranking court terme**.

    Horizon 4d

Pour `4d`, les métriques sont :

- `n_obs = 856863`
- `target_rate = 0.4943`
- `accuracy = 0.5099`
- `f1 = 0.4997`
- `roc_auc = 0.5085`
- `log_loss = 0.6949`
- `brier_score = 0.2508`

C’est l’horizon le moins convaincant. Le `ROC AUC` retombe presque exactement au niveau du hasard, ce qui suggère que le modèle distingue très peu les bons et mauvais cas à cet horizon. L’`accuracy` et le `F1` restent autour de `0.50`, donc le modèle ne s’effondre pas totalement, mais il ne fournit pas non plus de vraie puissance prédictive. Le `log loss` et le `Brier score` sont légèrement plus mauvais que pour `1d`, ce qui montre que les probabilités sont encore moins informatives. En pratique, cela suggère que l’horizon intermédiaire est probablement le plus difficile à modéliser avec un modèle de ce type.

    Horizon 24d

Pour `24d`, les métriques sont :

- `n_obs = 842954`
- `target_rate = 0.4796`
- `accuracy = 0.5155`
- `f1 = 0.5089`
- `roc_auc = 0.5151`
- `log_loss = 0.7048`
- `brier_score = 0.2554`

À `24d`, l’`accuracy` et le `F1` sont légèrement supérieurs à ceux des deux autres horizons, ce qui peut donner l’impression d’une amélioration. Toutefois, cette lecture doit être nuancée : le `ROC AUC` reste faible, donc la capacité de discrimination demeure limitée. Surtout, le `log loss` et le `Brier score` sont les plus mauvais des trois horizons, ce qui montre que les probabilités sont moins bien calibrées et moins fiables à long horizon. Autrement dit, le modèle peut parfois mieux classer certaines observations, mais il reste globalement peu sûr dans ses scores.

    Lecture d’ensemble

Pris ensemble, ces résultats suggèrent plusieurs points importants :

- le modèle est **légèrement meilleur que le hasard**, mais seulement de façon marginale ;
- `1d` semble être l’horizon le plus exploitable dans une logique de **ranking court terme** ;
- `4d` est l’horizon le plus faible ;
- `24d` montre un peu plus de structure en classification, mais avec des probabilités plus dégradées.

Dans le cadre du projet, cela reste intéressant, car l’objectif principal n’est pas seulement de prédire une classe 0/1, mais surtout de produire une **probabilité de succès** pour ensuite **classer les indicateurs** par `date × asset × horizon`. Sous cet angle, même un modèle globalement modeste peut déjà fournir une information utile s’il permet d’ordonner les signaux de façon un peu meilleure que le hasard.

    Conclusion

Ces résultats ne valident pas encore un modèle prédictif fort. En revanche, ils montrent que :

- le pipeline apprend quelque chose ;
- les trois horizons doivent bien être traités séparément ;
- et le modèle peut déjà servir de **benchmark interprétable** et de **mécanisme de ranking**.